# Atelier 2.B — Découpage et recherche hybride

**Durée : 30 min**

Quelle stratégie de découpage et quel mode de recherche retrouvent le mieux les bons extraits ? On ne devine pas : on **mesure**, sur le golden set de 32 questions (`donnees/golden_set.json`).

Cet atelier n'appelle aucun LLM : tout tourne en local, sans quota.

In [1]:
import pandas as pd
import rag

documents = rag.charger_corpus()
encodeur = rag.Encodeur()
QUESTIONS = [q for q in rag.charger_golden_set() if q["indices"]]  # questions dont la réponse est dans le corpus
K = 4                                                                # extraits transmis au modèle
print(len(QUESTIONS), "questions évaluées")

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


30 questions évaluées


## Les métriques

Pour chaque question, le golden set donne des **indices** : des passages qui doivent figurer dans les extraits retrouvés pour qu'on puisse répondre (par exemple « 8 heures ouvrées »).

- **Couverture** (rappel du contexte) : part des indices présents dans les `K` extraits retrouvés, en moyenne sur les questions.
- **Context precision@k** : les extraits pertinents (qui contiennent un indice) sont-ils **en tête** ? Moyenne des precision@i aux rangs pertinents, comme dans RAGAS : 1 si tous les extraits pertinents sont en tête.
- **MRR** (*mean reciprocal rank*) : 1 si le premier extrait est pertinent, 1/2 si c'est le deuxième, etc.

L'atelier 2.C calcule les mêmes métriques avec un LLM juge (bibliothèque RAGAS).

In [2]:
def mesurer(index, mode="dense", k=K):
    couvertures, precisions, inverses_rang = [], [], []
    for q in QUESTIONS:
        trouves = index.rechercher(q["question"], k, mode)
        couvertures.append(rag.couverture(trouves, q["indices"]))
        precisions.append(rag.precision_moyenne(trouves, q["indices"]))
        rang = rag.rang_premier_pertinent(trouves, q["indices"])
        inverses_rang.append(1 / rang if rang else 0)
    n = len(QUESTIONS)
    return {"couverture": round(sum(couvertures) / n, 3),
            "context_precision": round(sum(precisions) / n, 3),
            "mrr": round(sum(inverses_rang) / n, 3)}

## 1. Stratégies de découpage

On compare la taille des extraits, le chevauchement, le découpage par sections et l'ajout du titre du document en tête d'extrait. La colonne `caracteres_transmis` estime ce que coûte le contexte envoyé au modèle (`K` × taille moyenne).

In [3]:
CONFIGURATIONS = {
    "fixe 800": dict(strategie="fixe", taille=800, chevauchement=0),
    "récursif 300": dict(strategie="recursif", taille=300, chevauchement=50),
    "récursif 800": dict(strategie="recursif", taille=800, chevauchement=100),
    "récursif 1500": dict(strategie="recursif", taille=1500, chevauchement=150),
    "sections 800": dict(strategie="sections", taille=800, chevauchement=100),
    "sections 800 + entête": dict(strategie="sections", taille=800, chevauchement=100, entete=True),
}

lignes, index_par_config = [], {}
for nom, config in CONFIGURATIONS.items():
    extraits = rag.decouper(documents, **config)
    index = rag.Index(extraits, encodeur)
    index_par_config[nom] = index
    taille = sum(len(e.texte) for e in extraits) / len(extraits)
    lignes.append({"configuration": nom, "extraits": len(extraits), "taille_moyenne": round(taille),
                   "caracteres_transmis": round(K * taille), **mesurer(index)})

pd.DataFrame(lignes).set_index("configuration")

,extraits,taille_moyenne,caracteres_transmis,couverture,context_precision,mrr
configuration,,,,,,
fixe 800,32,675,2702,0.867,0.572,0.578
récursif 300,99,224,898,0.833,0.675,0.689
récursif 800,38,597,2390,0.867,0.625,0.631
récursif 1500,24,934,3738,0.767,0.561,0.561
sections 800,67,331,1323,0.867,0.711,0.717
sections 800 + entête,67,420,1681,0.900,0.767,0.786


**Questions**
1. Les grands extraits obtiennent-ils une meilleure couverture ? À quel prix en contexte transmis ?
2. Que change l'en-tête (titre du document) ajouté à chaque extrait ?
3. Le découpage fixe coupe au milieu des phrases : cela se voit-il dans les mesures ?

## 2. Les questions qui échouent

Pour la configuration retenue, quelles questions n'ont pas tous leurs indices dans les extraits retrouvés ?

In [4]:
CONFIG = "sections 800 + entête"
index = index_par_config[CONFIG]

for q in QUESTIONS:
    trouves = index.dense(q["question"], K)
    if rag.couverture(trouves, q["indices"]) < 1:
        print(f"{q['id']} [{q['type']}] {q['question']}")
        print("   attendu dans :", q["sources"], "| indices :", q["indices"])
        print("   retrouvé     :", [e.source for e in trouves], "\n")

Q17 [calcul] Combien coûtent 200 ramettes de papier recyclé PA-4R ?
   attendu dans : ['catalogue_produits.docx'] | indices : ['5.90 CHF la ramette à partir de 100 ramettes']
   retrouvé     : ['cgv_fictiva.pdf', 'politique_retours.md', 'politique_retours.md', 'conditions_maintenance.pdf'] 

Q18 [tableau] Quel est le délai d'intervention du contrat de maintenance Argent ?
   attendu dans : ['conditions_maintenance.pdf'] | indices : ['8 heures ouvrées']
   retrouvé     : ['conditions_maintenance.pdf', 'cgv_fictiva.pdf', 'guide_livraison.md', 'conditions_maintenance.pdf'] 

Q29 [factuelle] L'authentification à deux facteurs est-elle obligatoire sur le portail client ?
   attendu dans : ['conditions_portail.md'] | indices : ['obligatoire pour les comptes administrateurs']
   retrouvé     : ['conditions_portail.md', 'conditions_portail.md', 'cgv_fictiva.pdf', 'cgv_fictiva.pdf'] 



## 3. Recherche dense, lexicale, hybride

- **Dense** : similarité des embeddings ; comprend les reformulations et les synonymes.
- **Lexicale** (BM25) : mots communs, pondérés par leur rareté ; excelle sur les références exactes (« EN 1335 », « NS-4 »).
- **Hybride** : fusion des deux classements par *Reciprocal Rank Fusion* : chaque extrait reçoit la somme de 1 / (60 + rang) sur les deux classements.

In [5]:
modes = {mode: mesurer(index, mode) for mode in ("dense", "lexical", "hybride")}
pd.DataFrame(modes).T

,couverture,context_precision,mrr
dense,0.900,0.767,0.786
lexical,0.767,0.653,0.656
hybride,0.900,0.700,0.717


In [6]:
def couverture_par_type(mode):
    resultats = {}
    for q in QUESTIONS:
        resultats.setdefault(q["type"], []).append(rag.couverture(index.rechercher(q["question"], K, mode), q["indices"]))
    return {t: round(sum(v) / len(v), 2) for t, v in resultats.items()}

pd.DataFrame({mode: couverture_par_type(mode) for mode in ("dense", "lexical", "hybride")})

,dense,lexical,hybride
factuelle,0.94,0.76,0.88
multi-sources,1.00,1.00,1.00
tableau,0.83,0.67,0.83
calcul,0.50,1.00,1.00
interne,1.00,0.00,1.00


## 4. Là où les modes divergent

In [7]:
for q in QUESTIONS:
    dense = rag.couverture(index.dense(q["question"], K), q["indices"])
    lexical = rag.couverture(index.lexical(q["question"], K), q["indices"])
    if dense != lexical:
        gagnant = "dense" if dense > lexical else "lexical"
        print(f"{gagnant:8} l'emporte | {q['id']} {q['question']}  (dense {dense:.2f}, lexical {lexical:.2f})")

dense    l'emporte | Q07 Quel tribunal est compétent en cas de litige avec Fictiva ?  (dense 1.00, lexical 0.00)
dense    l'emporte | Q11 En combien de temps êtes-vous livrés à Zurich ?  (dense 1.00, lexical 0.00)
dense    l'emporte | Q13 Livrez-vous en France ?  (dense 1.00, lexical 0.00)
dense    l'emporte | Q16 Vos sièges ergonomiques sont-ils certifiés ?  (dense 1.00, lexical 0.00)
lexical  l'emporte | Q17 Combien coûtent 200 ramettes de papier recyclé PA-4R ?  (dense 0.00, lexical 1.00)
lexical  l'emporte | Q18 Quel est le délai d'intervention du contrat de maintenance Argent ?  (dense 0.00, lexical 1.00)
dense    l'emporte | Q23 Sous quel délai suis-je remboursé après un retour ?  (dense 1.00, lexical 0.00)
dense    l'emporte | Q28 Combien de temps conservez-vous les données d'un prospect qui n'a jamais commandé ?  (dense 1.00, lexical 0.00)
lexical  l'emporte | Q29 L'authentification à deux facteurs est-elle obligatoire sur le portail client ?  (dense 0.00, lexical 1.00)
dense  

✏️ Ajoutez au jeu de référence deux questions qui contiennent une référence produit exacte (par exemple « Le DK-200 gère-t-il deux écrans 4K ? ») et deux questions formulées sans aucun mot du corpus. Quel mode les retrouve ?

## À retenir

- Le découpage se **mesure** : taille, chevauchement, sections et en-têtes changent la couverture, et le coût du contexte transmis.
- Le dense comprend le sens, le lexical les références exactes. L'hybride n'est pas automatiquement meilleur : il dépend du corpus et des questions.
- Un jeu de questions de référence, avec ce qu'il faut retrouver, est l'outil de base de toute amélioration.